# B007: VSA Noise Resilience Analysis

**Trinity S³AI Framework — Zenodo v6.2**

This notebook analyzes the Vector Symbolic Architecture (VSA) operations:
- Noise resilience across different noise levels
- Retrieval accuracy degradation
- SIMD speedup benchmarks
- Binding/unbundling/bundling operations

---

**φ² + 1/φ² = 3 | TRINITY**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

DATA_PATH = Path('../data/B007_noise_resilience.csv')

## 1. Load Noise Resilience Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} noise level measurements")
print(f"\nColumns: {list(df.columns)}")
df.head()

## 2. Noise Resilience Curve

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(df['noise_percent'], df['accuracy'], 'o-', linewidth=2, markersize=8, label='VSA Retrieval')
ax.fill_between(df['noise_percent'],
                df['accuracy_lower'],
                df['accuracy_upper'],
                alpha=0.3)

# Baseline (random)
ax.axhline(y=1.0/1000, color='r', linestyle='--', alpha=0.5, label='Random Baseline')

ax.set_xlabel('Noise Percent (%)', fontsize=12)
ax.set_ylabel('Retrieval Accuracy', fontsize=12)
ax.set_title('B007: VSA Noise Resilience (Ternary {-1,0,+1})', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)

# Annotate key points
for _, row in df[df['noise_percent'].isin([10, 30, 50])].iterrows():
    ax.annotate(f"{row['accuracy']:.3f}",
                (row['noise_percent'], row['accuracy']),
                textcoords="offset points",
                xytext=(0,10), ha='center')

plt.tight_layout()
plt.savefig('../figures/B007_noise_resilience_analysis.png', dpi=300)
plt.show()

print(f"\nAt 50%% noise: accuracy = {df[df['noise_percent']==50]['accuracy'].values[0]:.4f}")

## 3. SIMD Speedup Analysis

In [ ]:
# SIMD benchmark data (from v6.2)
operations = ['Bind', 'Bundle', 'Cosine', 'Permute']
scalar_ns = [45, 52, 68, 38]
simd_ns = [3.2, 4.4, 4.0, 2.8]

speedup = [s/v for s, v in zip(scalar_ns, simd_ns)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Absolute times
x = np.arange(len(operations))
width = 0.35

ax1.bar(x - width/2, scalar_ns, width, label='Scalar', alpha=0.8)
ax1.bar(x + width/2, simd_ns, width, label='SIMD (NEON)', alpha=0.8)
ax1.set_ylabel('Time (ns)', fontsize=12)
ax1.set_title('B007: Absolute Runtime', fontsize=13, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(operations)
ax1.legend(fontsize=11)
ax1.set_yscale('log')
ax1.grid(True, alpha=0.3, axis='y')

# Speedup
bars = ax2.bar(x, speedup, color='steelblue', alpha=0.8)
ax2.set_ylabel('Speedup (×)', fontsize=12)
ax2.set_title('B007: SIMD Speedup', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(operations)
ax2.axhline(y=10, color='r', linestyle='--', alpha=0.5, label='10×')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars, speedup):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}×', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../figures/B007_simd_speedup_analysis.png', dpi=300)
plt.show()

print(f"\nAverage SIMD speedup: {np.mean(speedup):.1f}×")
print(f"Max speedup: {max(speedup):.1f}× ({operations[speedup.index(max(speedup))]})")

## 4. Operation Complexity Analysis

In [ ]:
# Theoretical vs actual complexity
complexity_data = {
    'Operation': ['Bind', 'Unbind', 'Bundle2', 'Bundle3', 'Cosine', 'Permute'],
    'Theoretical': ['O(n)', 'O(n)', 'O(n)', 'O(n)', 'O(n)', 'O(n)'],
    'Actual (ns/op)': [3.2, 3.5, 4.4, 5.8, 4.0, 2.8],
    'Vector Dimension': [1024, 1024, 1024, 1024, 1024, 1024]
}

complexity_df = pd.DataFrame(complexity_data)
print("VSA Operation Complexity:")
print(complexity_df.to_string(index=False))

# Calculate operations per second
complexity_df['M ops/sec'] = 1000 / complexity_df['Actual (ns/op)']
print(f"\nOperations per second:")
print(complexity_df[['Operation', 'M ops/sec']].to_string(index=False))

## 5. Calibration Metrics

In [ ]:
# VSA calibration (from v6.2)
ece_min = 0.058
ece_max = 0.072
brier_min = 0.162
brier_max = 0.185

print("VSA Calibration Metrics:")
print(f"  ECE: {ece_min:.3f} - {ece_max:.3f}")
print(f"  Brier Score: {brier_min:.3f} - {brier_max:.3f}")

interpretation = "Excellent-Good"
print(f"\nInterpretation: {interpretation}")
print(f"  (ECE < 0.1 = Well-calibrated)")

## 6. Summary

In [ ]:
print("="*60)
print("B007: VSA Analysis Summary")
print("="*60)

print(f"\nNoise Resilience:")
for _, row in df.iterrows():
    print(f"  {row['noise_percent']:3.0f}% noise: {row['accuracy']:.4f} accuracy")

print(f"\nSIMD Performance:")
print(f"  Average speedup: {np.mean(speedup):.1f}×")
print(f"  Max speedup: {max(speedup):.1f}×")

print(f"\nCalibration:")
print(f"  ECE: {ece_min:.3f} - {ece_max:.3f} ({interpretation})")

print("="*60)